# RAG Assignment: Baseline Dense Retrieval vs. Cross-Encoder Re-Ranking

**Document:** NIST SP 800-63B, *Digital Identity Guidelines: Authentication and Lifecycle Management* (June 2017, includes updates as of 2020-03-02).
It is a long normative standard (80 pages) full of `SHALL` / `SHALL NOT` rules, exceptions and level-specific limits (AAL1/2/3), which makes it a good stress test for retrieval.
*Note:* NIST withdrew this edition on 2025-08-01 in favour of SP 800-63B-4. That does not matter for a retrieval experiment, but answers below describe the 2017/2020 text only.

**Pipelines compared (same corpus, same Top-10 candidates, same LLM and prompt):**

| | Pipeline 1 – Baseline | Pipeline 2 – Re-ranking |
|---|---|---|
| Stage 1 | Bi-encoder (`all-MiniLM-L6-v2`) + FAISS inner product → Top 10 | *same Top 10* |
| Stage 2 | keep dense ranks 1–3 | Cross-encoder (`ms-marco-MiniLM-L-6-v2`) scores all 10 pairs → sort desc → Top 3 |
| Generation | Gemini (`gemini-3.1-flash-lite`), identical prompt → **Answer 1** | same → **Answer 2** |

Notebook outline: 1 setup · 2 PDF extraction · 3 chunking · 4 embeddings & index · 5 baseline retrieval · 6 re-ranking · 7 answer generation · 8 failure-case & rank-shift analysis · 9 latency · 10 conclusions & limitations.

## 1. Environment setup and imports

All models except the LLM run locally on CPU. The Gemini API key is read from the environment variable `GEMINI_API_KEY` (or a git-ignored `.env` file) — it is never written into this notebook.

In [1]:
import os, re, json, time, platform, statistics
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import faiss
import pymupdf                      # PyMuPDF, for PDF text extraction
import requests
import sentence_transformers
from sentence_transformers import SentenceTransformer, CrossEncoder

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

# ---- Experiment configuration (everything that must stay identical across pipelines) ----
PDF_PATH         = Path("data/NIST.SP.800-63b.pdf")
FIRST_BODY_PAGE  = 12      # 1-based PDF page where Section 1 starts; pages 1-11 are cover, withdrawal notice, TOC, errata
MAX_CHUNK_TOKENS = 200     # measured with the bi-encoder tokenizer (model limit is 256 word pieces -> no truncation)
OVERLAP_TOKENS   = 40      # trailing sentences (<= 40 tokens) repeated at the start of the next chunk in the same section
BI_ENCODER_NAME  = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
TOP_K = 10                 # candidates from dense retrieval
TOP_N = 3                  # chunks given to the LLM
LLM_MODEL = "gemini-3.1-flash-lite"   # free tier allows only ~20 requests/day for gemini-3.8-flash; a run needs 32
GENERATION_CONFIG = {"temperature": 0.0, "maxOutputTokens": 1024, "thinkingConfig": {"thinkingLevel": "low"}}
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)

torch.manual_seed(0); np.random.seed(0)
print("python", platform.python_version(), "| torch", torch.__version__, "| sentence-transformers", sentence_transformers.__version__,
      "| faiss", faiss.__version__, "| pymupdf", pymupdf.__version__)
print("CPU:", platform.processor() or platform.machine(), "| logical cores:", os.cpu_count(), "| torch threads:", torch.get_num_threads(),
      "| CUDA:", torch.cuda.is_available())

python 3.12.3 | torch 2.14.0+cpu | sentence-transformers 6.1.0 | faiss 1.15.1 | pymupdf 1.28.2
CPU: x86_64 | logical cores: 12 | torch threads: 6 | CUDA: False


In [2]:
# Load GEMINI_API_KEY from a local .env file if it is not already set (tiny parser, avoids an extra dependency).
if "GEMINI_API_KEY" not in os.environ and Path(".env").exists():
    for line in Path(".env").read_text().splitlines():
        if "=" in line and not line.lstrip().startswith("#"):
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
HAVE_LLM = bool(os.environ.get("GEMINI_API_KEY"))
print("GEMINI_API_KEY found:", HAVE_LLM, "(retrieval and latency cells still run without it; generation is skipped)")

GEMINI_API_KEY found: True (retrieval and latency cells still run without it; generation is skipped)


## 2. PDF loading and text extraction

* Every page starts with the same running header (title lines, page number, "This publication is available free of charge…"). Everything up to and including that line is dropped.
* Section headings come from the PDF's own bookmark outline (`get_toc()`), because Word's automatic heading numbers (e.g. "4.1") are not in the extracted text. Each heading line found on its page is tagged so that every chunk can be traced back to its section.

In [3]:
if not PDF_PATH.exists():
    raise FileNotFoundError(f"{PDF_PATH} missing - download it from https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-63b.pdf")

doc = pymupdf.open(PDF_PATH)
HEADER_END = "This publication is available free of charge"

def page_lines(page):
    # Return the non-empty text lines of a page with the running header removed.
    raw = [l.strip() for l in page.get_text().splitlines()]
    end = next((i for i, l in enumerate(raw) if l.startswith(HEADER_END)), -1)
    return [l for l in raw[end + 1:] if l]

# Outline entries: (level, "5.1.1.2 Memorized Secret Verifiers", page). Split into number and name.
toc_by_page = {}
for level, title, page in doc.get_toc():
    title = re.sub(r"\s+", " ", title).strip()
    m = re.match(r"^((?:\d+|[A-Z])(?:\.\d+)*)\s+(.*)$", title)
    num, name = (m.group(1), m.group(2)) if m else ("", title)
    toc_by_page.setdefault(page, []).append({"title": title, "num": num, "name": name})

pages = {p: page_lines(doc[p - 1]) for p in range(FIRST_BODY_PAGE, len(doc) + 1)}

# Sanity checks on extraction: no page should be empty / unreadable, and header removal must have worked.
empty = [p for p, ls in pages.items() if len(" ".join(ls)) < 100]
assert not empty, f"pages with (almost) no extractable text: {empty}"
assert not any(l.startswith(HEADER_END) for ls in pages.values() for l in ls), "running header not removed"
print(f"PDF pages: {len(doc)} | indexed pages: {FIRST_BODY_PAGE}-{len(doc)} ({len(pages)} pages) | "
      f"characters: {sum(len(' '.join(ls)) for ls in pages.values()):,} | outline entries: {sum(map(len, toc_by_page.values()))}")
print("Example (page 21, first lines):", pages[21][:4])

PDF pages: 80 | indexed pages: 12-80 (69 pages) | characters: 162,654 | outline entries: 120
Example (page 21, first lines): ['When a biometric factor is used in authentication at AAL3, the verifier SHALL make a', 'determination that the biometric sensor and subsequent processing meet the performance', 'requirements stated in Section 5.2.3.', '4.3.3 Reauthentication']


## 3. Chunking and chunk identification

1. Lines are joined into running text and split into sentence-like *units* (split after `.`, `;`, `:` when the next token starts a new sentence/bullet).
2. A heading always starts a new chunk, so a chunk never mixes two sections (consecutive headings such as "5.1" + "5.1.1" are merged).
3. Units are packed greedily up to **200 bi-encoder tokens**; the last ≤40 tokens of sentences are repeated at the start of the next chunk of the same section (overlap), so a rule split across a boundary is still seen whole once.
4. Each chunk gets a stable ID `C000…`, plus its section title and PDF page range — these IDs are carried through every later step.

In [4]:
bi_encoder = SentenceTransformer(BI_ENCODER_NAME, device="cpu")
tok = bi_encoder.tokenizer
n_tokens = lambda s: len(tok.tokenize(s))   # only used to *count* tokens; a "longer than max length" warning for raw
                                            # table text is expected here - such units are split before embedding
SENT_SPLIT = re.compile(r"(?<=[.;:])\s+(?=[A-Z•(\[])")

def norm(s):
    # Compare headings ignoring all whitespace: the outline says "Appendix A— Strength", the page text "Appendix A—Strength".
    return re.sub(r"\s+", "", s).lower()

def extract_units(pages, toc_by_page):
    # Turn page lines into sentence units tagged with page, section, and whether the unit is a heading.
    units, buf, buf_page, section = [], "", None, "(front)"

    def flush(final):
        nonlocal buf, buf_page
        parts = SENT_SPLIT.split(buf) if buf else []
        keep = parts if final else parts[:-1]
        for p in keep:
            if p.strip():
                units.append({"page": buf_page, "section": section, "text": p.strip(), "heading": False})
        buf = "" if final or not parts else parts[-1]

    for p, lines in pages.items():
        pending = list(toc_by_page.get(p, []))          # headings expected on this page, in order
        i = 0
        while i < len(lines):
            line = lines[i]; i += 1
            head = None
            if pending:
                h = pending[0]
                if norm(line) in (norm(h["name"]), norm(h["title"])):
                    head = h
                elif norm(line) == norm(h["num"]) and i < len(lines) and norm(lines[i]) == norm(h["name"]):
                    head = h; i += 1                     # number printed on its own line before the name
            if head:
                pending.pop(0)
                flush(final=True)
                section = head["title"]
                units.append({"page": p, "section": section, "text": head["title"], "heading": True})
                continue
            if not buf:
                buf_page = p
            # Join wrapped lines; keep hyphens at line ends ("out-of-" + "band" -> "out-of-band").
            buf = line if not buf else (buf + line if buf.endswith("-") else buf + " " + line)
            before = len(units)
            flush(final=False)
            if len(units) > before:
                buf_page = p
        missing = [h["title"] for h in pending]
        if missing:
            print(f"  note: outline heading(s) not found as a text line on page {p}: {missing}")
    flush(final=True)
    return units

def split_long(text, limit):
    # Split an over-long unit (mostly table rows / reference lists) into word windows of <= limit tokens.
    out, cur = [], []
    for w in text.split():
        if cur and n_tokens(" ".join(cur + [w])) > limit:
            out.append(" ".join(cur)); cur = []
        cur.append(w)
    if cur: out.append(" ".join(cur))
    return out

def make_chunks(units, max_tokens=MAX_CHUNK_TOKENS, overlap=OVERLAP_TOKENS):
    pieces = []
    for u in units:
        texts = split_long(u["text"], max_tokens - overlap) if n_tokens(u["text"]) > max_tokens - overlap else [u["text"]]
        for t in texts:
            pieces.append({**u, "text": t, "ntok": n_tokens(t)})
    chunks, cur = [], []

    def emit():
        chunks.append({"chunk_id": f"C{len(chunks):03d}", "section": cur[0]["section"],
                       "page_start": cur[0]["page"], "page_end": cur[-1]["page"],
                       "text": " ".join(x["text"] for x in cur)})

    for u in pieces:
        size = sum(x["ntok"] for x in cur)
        only_headings = all(x["heading"] for x in cur)
        if cur and ((u["heading"] and not only_headings) or size + u["ntok"] > max_tokens):
            emit()
            carry = []
            if not u["heading"]:                         # overlap only inside the same section
                for x in reversed(cur):
                    if x["heading"] or sum(y["ntok"] for y in carry) + x["ntok"] > overlap: break
                    carry.insert(0, x)
            cur = carry
        cur.append(u)
    if cur: emit()
    return chunks

units = extract_units(pages, toc_by_page)
chunks = make_chunks(units)
chunk_by_id = {c["chunk_id"]: c for c in chunks}
for c in chunks:
    c["n_tokens"] = n_tokens(c["text"])

# Checks: unique IDs, nothing longer than the bi-encoder can see, every chunk has a section and page.
assert len(chunk_by_id) == len(chunks)
assert max(c["n_tokens"] for c in chunks) <= bi_encoder.max_seq_length, "a chunk would be truncated by the bi-encoder"
assert all(c["section"] != "(front)" and c["page_start"] >= FIRST_BODY_PAGE for c in chunks)
lens = [c["n_tokens"] for c in chunks]
print(f"{len(units)} units -> {len(chunks)} chunks | tokens per chunk: min {min(lens)}, median {int(np.median(lens))}, max {max(lens)} "
      f"(bi-encoder max_seq_length = {bi_encoder.max_seq_length})")
pd.DataFrame(chunks)[["chunk_id", "section", "page_start", "page_end", "n_tokens", "text"]].iloc[[0, 40, 41, 100, 200]]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (617 > 256). Running this sequence through the model will result in indexing errors


1223 units -> 254 chunks | tokens per chunk: min 8, median 167, max 200 (bi-encoder max_seq_length = 256)


,chunk_id,section,page_start,page_end,n_tokens,text
0,C000,1 Purpose,12,12,49,"1 Purpose This section is informative. This document and its companion documents, Special Publication (SP) 800-63, S..."
40,C040,4.5 Summary of Requirements,22,22,117,MAY use one authentication factor 12 hours or 15 minutes inactivity; SHALL use both authentication factors Requireme...
41,C041,5 Authenticator and Verifier Requirements,24,24,94,5 Authenticator and Verifier Requirements This section is normative. This section provides the detailed requirements...
100,C100,5.2.1 Physical Authenticators,36,36,62,5.2.1 Physical Authenticators CSPs SHALL provide subscriber instructions on how to appropriately protect the authent...
200,C200,10.2.3 Out-of-Band,65,65,184,10.2.3 Out-of-Band Typical Usage Out-of-band authentication requires users have access to a primary and secondary co...


In [5]:
# Save the exact corpus so every rank reported later can be looked up.
with open(RESULTS_DIR / "chunks.jsonl", "w") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")
print("wrote", RESULTS_DIR / "chunks.jsonl")

wrote results/chunks.jsonl


## 4. Dense embeddings and vector index (Bi-Encoder)

Each chunk is embedded **independently** by the bi-encoder into a 384-d vector. Vectors are L2-normalised, so the inner product used by `faiss.IndexFlatIP` equals cosine similarity. A flat index does exact (brute-force) search — fine for 246 chunks and it removes approximate-search error from the comparison.

In [6]:
t0 = time.perf_counter()
chunk_emb = bi_encoder.encode([c["text"] for c in chunks], batch_size=32, normalize_embeddings=True,
                              convert_to_numpy=True, show_progress_bar=False).astype("float32")
index_build_s = time.perf_counter() - t0

index = faiss.IndexFlatIP(chunk_emb.shape[1])
index.add(chunk_emb)
assert index.ntotal == len(chunks)
assert np.allclose(np.linalg.norm(chunk_emb, axis=1), 1.0, atol=1e-4)    # normalised -> IP == cosine
print(f"embedded {len(chunks)} chunks, dim={chunk_emb.shape[1]}, in {index_build_s:.2f}s (one-off, offline cost)")

embedded 254 chunks, dim=384, in 6.43s (one-off, offline cost)


## 5. Pipeline 1 – Baseline retrieval (Top 10 → original Top 3)

In [7]:
def dense_retrieve(query, k=TOP_K):
    # Stage 1 (shared by both pipelines): embed query, exact inner-product search, return Top-k in dense order.
    q = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, ids = index.search(q, k)
    cands = [{"chunk_id": chunks[i]["chunk_id"], "dense_rank": r + 1, "dense_score": float(s)}
             for r, (i, s) in enumerate(zip(ids[0], scores[0])) if i != -1]
    if len(cands) < k:
        print(f"WARNING: only {len(cands)} candidates returned for {query!r}")
    return cands

def baseline_select(cands, n=TOP_N):
    # Pipeline 1: keep the original dense order and take the first n.
    return cands[:n]

def full_dense_rank(query):
    # Rank of every chunk in the whole corpus (diagnostic only: tells us how far away a missed chunk was).
    q = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    _, ids = index.search(q, index.ntotal)
    return {chunks[i]["chunk_id"]: r + 1 for r, i in enumerate(ids[0])}

demo = dense_retrieve("Are password hints allowed?")
pd.DataFrame([{**c, "section": chunk_by_id[c["chunk_id"]]["section"]} for c in demo])

,chunk_id,dense_rank,dense_score,section
0,C198,1,0.569622,10.2.1 Memorized Secrets
1,C243,2,0.557299,Appendix A— Strength of Memorized Secrets
2,C244,3,0.553751,A.1 Introduction
3,C042,4,0.545551,5.1 Requirements by Authenticator Type
4,C251,5,0.535115,A.3 Complexity
5,C046,6,0.526016,5.1.1.2 Memorized Secret Verifiers
6,C199,7,0.515576,10.2.2 Look-Up Secrets
7,C050,8,0.510428,5.1.1.2 Memorized Secret Verifiers
8,C197,9,0.509628,10.2.1 Memorized Secrets
9,C249,10,0.501920,A.3 Complexity


## 6. Pipeline 2 – Cross-Encoder re-ranking (same Top 10 → re-sorted Top 3)

The cross-encoder reads the query and the chunk **together** (`[CLS] query [SEP] chunk [SEP]`) and outputs one relevance logit; **higher = more relevant** for this MS-MARCO model, so we sort descending. Python's sort is stable, so equal scores keep their dense order.

In [8]:
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device="cpu")
CE_MAX_LEN = cross_encoder.max_seq_length

def rerank(query, cands, n=TOP_N):
    # Pipeline 2: score ALL candidate (query, chunk) pairs, sort by score descending, return (full re-ranked list, top n).
    pairs = [(query, chunk_by_id[c["chunk_id"]]["text"]) for c in cands]
    scores = cross_encoder.predict(pairs, batch_size=len(pairs), show_progress_bar=False)
    scored = [{**c, "ce_score": float(s)} for c, s in zip(cands, scores)]
    ranked = sorted(scored, key=lambda c: c["ce_score"], reverse=True)
    for r, c in enumerate(ranked):
        c["rerank_rank"] = r + 1
    return ranked, ranked[:n]

# Truncation check: query + longest chunk must fit in the cross-encoder window.
longest = max(chunks, key=lambda c: len(cross_encoder.tokenizer(c["text"])["input_ids"]))
pair_len = len(cross_encoder.tokenizer("a long example question about authenticators and assurance levels?", longest["text"])["input_ids"])
assert pair_len <= CE_MAX_LEN, f"pair length {pair_len} > cross-encoder max {CE_MAX_LEN}"
print(f"cross-encoder max_length={CE_MAX_LEN}; longest query+chunk pair = {pair_len} tokens -> no truncation")

ranked_demo, _ = rerank("Are password hints allowed?", demo)
pd.DataFrame([{**c, "section": chunk_by_id[c["chunk_id"]]["section"]} for c in ranked_demo])

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

cross-encoder max_length=512; longest query+chunk pair = 214 tokens -> no truncation


,chunk_id,dense_rank,dense_score,ce_score,rerank_rank,section
0,C249,10,0.501920,-2.974736,1,A.3 Complexity
1,C244,3,0.553751,-3.599351,2,A.1 Introduction
2,C046,6,0.526016,-3.907215,3,5.1.1.2 Memorized Secret Verifiers
3,C251,5,0.535115,-4.143411,4,A.3 Complexity
4,C050,8,0.510428,-4.956090,5,5.1.1.2 Memorized Secret Verifiers
5,C243,2,0.557299,-5.309730,6,Appendix A— Strength of Memorized Secrets
6,C197,9,0.509628,-5.692620,7,10.2.1 Memorized Secrets
7,C198,1,0.569622,-5.854072,8,10.2.1 Memorized Secrets
8,C042,4,0.545551,-7.469771,9,5.1 Requirements by Authenticator Type
9,C199,7,0.515576,-10.723521,10,10.2.2 Look-Up Secrets


### Correctness checks for the two pipelines

In [9]:
def check_pipelines(query):
    cands = dense_retrieve(query)
    base = baseline_select(cands)
    ranked, rr = rerank(query, cands)
    ids = [c["chunk_id"] for c in cands]
    assert len(cands) == TOP_K and len(set(ids)) == TOP_K, "need 10 distinct candidates"
    assert [c["dense_rank"] for c in cands] == list(range(1, TOP_K + 1))
    assert all(a["dense_score"] >= b["dense_score"] - 1e-6 for a, b in zip(cands, cands[1:])), "baseline not in dense order"
    assert [c["chunk_id"] for c in base] == ids[:TOP_N], "baseline must be original top-3"
    assert sorted(c["chunk_id"] for c in ranked) == sorted(ids), "re-ranker must score exactly the same 10 candidates"
    assert all(a["ce_score"] >= b["ce_score"] for a, b in zip(ranked, ranked[1:])), "re-ranking must be descending"
    assert len(base) == len(rr) == TOP_N
    # Chunk IDs still point to the right text/score after sorting: recompute one CE score independently.
    c0 = ranked[0]
    s = float(cross_encoder.predict([(query, chunk_by_id[c0["chunk_id"]]["text"])], show_progress_bar=False)[0])
    assert abs(s - c0["ce_score"]) < 1e-3, "chunk id / score mismatch after sorting"
    # Determinism: stage 1 gives the same candidates when called again (both pipelines share them).
    assert [c["chunk_id"] for c in dense_retrieve(query)] == ids
    return True

print(all(check_pipelines(q) for q in ["Are password hints allowed?", "How long is an out-of-band authentication secret valid?"]))

True


## 7. LLM answer generation (identical prompt for both pipelines)

The only thing that differs between Answer 1 and Answer 2 is **which three chunks** fill the `{context}` slot (in each pipeline's own rank order). Model, temperature (0), thinking level, token limit and template are identical.

In [10]:
PROMPT_TEMPLATE = '''You are answering questions about NIST SP 800-63B (Digital Identity Guidelines: Authentication and Lifecycle Management).
Use ONLY the context passages below; do not use outside knowledge.
If the passages do not contain the answer, reply exactly: "The provided context does not answer this question."
Keep normative keywords (SHALL, SHALL NOT, SHOULD, SHOULD NOT, MAY) as written, and cite the passage IDs you rely on in square brackets, e.g. [C012].
Answer in at most 4 sentences.

Context:
{context}

Question: {question}
Answer:'''

def build_prompt(question, selected):
    blocks = []
    for c in selected:
        ch = chunk_by_id[c["chunk_id"]]
        pages_ = f"p. {ch['page_start']}" + (f"-{ch['page_end']}" if ch["page_end"] != ch["page_start"] else "")
        blocks.append(f"[{ch['chunk_id']}] (Section {ch['section']}; PDF {pages_})\n{ch['text']}")
    return PROMPT_TEMPLATE.format(context="\n\n".join(blocks), question=question)

GEMINI_URL = f"https://generativelanguage.googleapis.com/v1beta/models/{LLM_MODEL}:generateContent"
CACHE_PATH = RESULTS_DIR / "llm_cache.json"     # answers keyed by hash(model, config, prompt): re-runs and quota limits don't force new calls
llm_cache = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}

def cache_key(prompt):
    import hashlib
    return hashlib.sha256(json.dumps([LLM_MODEL, GENERATION_CONFIG, prompt], sort_keys=True).encode()).hexdigest()

def generate(prompt, max_retries=6):
    # Call Gemini (with retries on overload / per-minute limits). Returns text, measured latency, model version, error.
    key = cache_key(prompt)
    if key in llm_cache:
        return {**llm_cache[key], "cached": True}
    if not HAVE_LLM:
        return {"text": None, "error": "GEMINI_API_KEY not set", "latency_s": None}
    body = {"contents": [{"role": "user", "parts": [{"text": prompt}]}], "generationConfig": GENERATION_CONFIG}
    last_err = None
    for attempt in range(1, max_retries + 1):
        t0 = time.perf_counter()
        try:
            r = requests.post(GEMINI_URL, json=body, timeout=120,
                              headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]})
            latency = time.perf_counter() - t0
            if r.status_code == 200:
                j = r.json()
                parts = j["candidates"][0].get("content", {}).get("parts", [])
                text = "".join(p.get("text", "") for p in parts if not p.get("thought")).strip()
                out = {"text": text, "latency_s": latency, "model_version": j.get("modelVersion"),
                       "finish_reason": j["candidates"][0].get("finishReason"), "attempts": attempt, "error": None}
                llm_cache[key] = out
                CACHE_PATH.write_text(json.dumps(llm_cache, indent=1, ensure_ascii=False))
                return {**out, "cached": False}
            last_err = f"HTTP {r.status_code}: {r.text[:300]}"
            if r.status_code not in (429, 500, 502, 503, 504) or "PerDay" in r.text:
                break                                   # not retryable (bad request, auth, daily quota exhausted)
        except requests.RequestException as e:
            last_err = repr(e)
        time.sleep(min(60, 2 ** attempt))
    print("Generation failed:", last_err[:300])
    return {"text": None, "latency_s": None, "error": last_err, "attempts": attempt}

# Structural check: both prompts share the same template; only the context block differs.
_c = dense_retrieve("test question")
p1, p2 = build_prompt("Q?", baseline_select(_c)), build_prompt("Q?", rerank("Q?", _c)[1])
strip_ctx = lambda p: p.split("Context:")[0] + p.split("Question:")[1]
assert strip_ctx(p1) == strip_ctx(p2)
print(build_prompt("Are password hints allowed?", baseline_select(demo))[:1200], "...")

You are answering questions about NIST SP 800-63B (Digital Identity Guidelines: Authentication and Lifecycle Management).
Use ONLY the context passages below; do not use outside knowledge.
If the passages do not contain the answer, reply exactly: "The provided context does not answer this question."
Keep normative keywords (SHALL, SHALL NOT, SHOULD, SHOULD NOT, MAY) as written, and cite the passage IDs you rely on in square brackets, e.g. [C012].
Answer in at most 4 sentences.

Context:
[C198] (Section 10.2.1 Memorized Secrets; PDF p. 64-65)
Encourage users to make memorized secrets as lengthy as they want, using any characters they like (including spaces), thus aiding memorization. o Do not impose other composition rules (e.g. mixtures of different character types) on memorized secrets. o Do not require that memorized secrets be changed arbitrarily (e.g., periodically) unless there is a user request or evidence of authenticator compromise. (See Section 5.1.1 for additional information

## 8. Failure-mode evaluation and rank-shift analysis

**How the queries were built.** I read the document and wrote questions in the way a user would ask them (often with everyday words such as *password*, *SMS*, *fingerprint*, whereas the standard says *memorized secret*, *PSTN*, *biometric*). For each query I identified, by reading the text, the sentence that answers it and recorded a short **key phrase** from that sentence; a chunk counts as *critical* if it contains one of the key phrases (checked against the chunk text below, so labels are not hand-assigned per rank). Queries were refined after looking at retrieval behaviour while exploring, and the set deliberately also contains cases where re-ranking does **not** help, so the results are not a curated success story.

`answer_check` is a crude regex for the key fact the correct answer must state; it is a quick screen, not a substitute for reading the answers (done in §8.3).

In [11]:
QUERIES = [
    # --- Keyword trap / distractor: user vocabulary ("password", "SMS") vs. document vocabulary ("memorized secret", "PSTN");
    #     Appendix A and the usability section talk *about* passwords a lot but are informative, not the rule.
    dict(qid="Q01", category="keyword trap", query="Are password hints allowed?",
         key_phrases=["SHALL NOT permit the subscriber to store a"], answer_check=r"SHALL NOT|not (be )?(allowed|permitted)"),
    dict(qid="Q02", category="keyword trap", query="How often should users be forced to change their passwords?",
         key_phrases=["changed arbitrarily"], answer_check=r"compromise"),
    dict(qid="Q03", category="keyword trap", query="What is the minimum password length?",
         key_phrases=["at least 8 characters in length"], answer_check=r"\b8\b|eight"),
    dict(qid="Q04", category="keyword trap", query="Can SMS text messages be used to deliver one-time codes?",
         key_phrases=["PSTN for out-of-band verification is RESTRICTED", "via the PSTN (SMS or voice)"], answer_check=r"(?i)restricted"),
    # --- Exceptions & corner cases: a general rule chunk vs. a specific limit / exception stated elsewhere.
    dict(qid="Q05", category="exception", query="How long is an out-of-band authentication secret valid?",
         key_phrases=["not completed within 10 minutes"], answer_check=r"10 minutes"),
    dict(qid="Q06", category="exception", query="Is a 6-character memorized secret ever acceptable?",
         key_phrases=["SHALL be at least 6 characters in length"], answer_check=r"(?i)random"),
    dict(qid="Q07", category="exception", query="How many failed fingerprint attempts are allowed before biometric authentication is disabled?",
         key_phrases=["no more than 5 consecutive failed"], answer_check=r"\b5\b|five"),
    dict(qid="Q08", category="exception", query="Which authenticators are permitted at AAL3?",
         key_phrases=["AAL3 authentication SHALL occur by the use of one of a combination"], answer_check=r"(?i)multi-factor cryptographic device"),
    # --- Negation & constraints: "SHALL NOT", "not considered", role/level-specific limits.
    dict(qid="Q09", category="negation", query="Are hashed low-complexity passwords considered verifier-compromise resistant?",
         key_phrases=["SHALL NOT be considered verifier compromise resistant"], answer_check=r"(?i)not (be )?considered|SHALL NOT|\bno\b"),
    dict(qid="Q10", category="negation", query="Does unlocking my phone with a PIN count as an authentication factor?",
         key_phrases=["SHALL NOT be considered one of the authentication factors"], answer_check=r"(?i)SHALL NOT|does not|not (be )?(count|considered)|\bno\b"),
    dict(qid="Q11", category="negation", query="Is email an acceptable out-of-band authentication method?",
         key_phrases=["or email, SHALL NOT be used"], answer_check=r"(?i)SHALL NOT|\bnot\b|\bno\b"),
    dict(qid="Q12", category="negation", query="Can session cookies survive a browser restart?",
         key_phrases=["SHALL NOT be retained across a restart"], answer_check=r"(?i)SHALL NOT|non-persistent|not be retained|\bno\b"),
    dict(qid="Q13", category="role/level limit", query="Can a password alone satisfy AAL2?",
         key_phrases=["combination of two single-factor authenticators"], answer_check=r"(?i)\bno\b|\bnot\b|two"),
    dict(qid="Q14", category="role/level limit", query="After how many minutes of inactivity must an AAL3 session be reauthenticated?",
         key_phrases=["inactivity lasting 15 minutes"], answer_check=r"15 minutes"),
    # --- Other document-specific: the answer sits in a long, generic-looking verifier section.
    dict(qid="Q15", category="buried detail", query="Can push notifications on a smartphone be used for out-of-band authentication?",
         key_phrases=["MAY send a push notification"], answer_check=r"(?i)push notification"),
    dict(qid="Q16", category="buried detail", query="What false match rate must a biometric system achieve?",
         key_phrases=["FMR [ISO/IEC 2382-37] of 1 in 1000"], answer_check=r"1 in 1,?000|0\.1 ?%"),
]

def critical_ids(qd):
    return [c["chunk_id"] for c in chunks if any(k.lower() in c["text"].lower() for k in qd["key_phrases"])]

for qd in QUERIES:
    qd["critical"] = critical_ids(qd)
    assert qd["critical"], f"{qd['qid']}: key phrase not found in any chunk - fix the label"
pd.DataFrame(QUERIES)[["qid", "category", "query", "critical"]]

,qid,category,query,critical
0,Q01,keyword trap,Are password hints allowed?,[C046]
1,Q02,keyword trap,How often should users be forced to change their passwords?,"[C049, C198]"
2,Q03,keyword trap,What is the minimum password length?,"[C043, C044]"
3,Q04,keyword trap,Can SMS text messages be used to deliver one-time codes?,"[C061, C062, C068]"
4,Q05,exception,How long is an out-of-band authentication secret valid?,[C066]
5,Q06,exception,Is a 6-character memorized secret ever acceptable?,"[C043, C046]"
6,Q07,exception,How many failed fingerprint attempts are allowed before biometric authentication is disabled?,[C107]
7,Q08,exception,Which authenticators are permitted at AAL3?,[C028]
8,Q09,negation,Are hashed low-complexity passwords considered verifier-compromise resistant?,[C117]
9,Q10,negation,Does unlocking my phone with a PIN count as an authentication factor?,[C022]


### 8.1 Run both pipelines on every query

Stage 1 is executed **once** per query and its Top-10 list is passed to both pipelines, so they cannot see different candidates.

In [12]:
runs, rank_rows = [], []
for qd in QUERIES:
    q = qd["query"]
    cands = dense_retrieve(q)                       # shared Top-10
    base_top = baseline_select(cands)               # Pipeline 1
    ranked, rr_top = rerank(q, cands)               # Pipeline 2
    full_rank = full_dense_rank(q)
    rr_rank = {c["chunk_id"]: c["rerank_rank"] for c in ranked}
    ce = {c["chunk_id"]: c["ce_score"] for c in ranked}
    for c in ranked:
        rank_rows.append({"qid": qd["qid"], "chunk_id": c["chunk_id"], "section": chunk_by_id[c["chunk_id"]]["section"],
                          "dense_rank": c["dense_rank"], "dense_score": round(c["dense_score"], 4),
                          "ce_score": round(c["ce_score"], 3), "rerank_rank": c["rerank_rank"],
                          "critical": c["chunk_id"] in qd["critical"]})
    crit = qd["critical"]
    best_dense = min(full_rank[c] for c in crit)
    in_top10 = [c for c in crit if c in rr_rank]
    runs.append({**qd, "cands": cands, "base_top": base_top, "rr_top": rr_top, "ranked": ranked,
                 "critical_dense_rank": {c: full_rank[c] for c in crit},
                 "critical_rerank_rank": {c: rr_rank.get(c) for c in crit},
                 "critical_ce": {c: round(ce[c], 2) for c in in_top10},
                 "base_hit": any(c["chunk_id"] in crit for c in base_top),
                 "rr_hit": any(c["chunk_id"] in crit for c in rr_top),
                 "best_dense_rank": best_dense})
    check_pipelines(q)

def outcome(r):
    if r["base_hit"] and r["rr_hit"]:   return "both hit"
    if r["rr_hit"]:                     return "RE-RANK RESCUE"
    if r["base_hit"]:                   return "RE-RANK LOSS"
    return "both miss (not in top 10)" if r["best_dense_rank"] > TOP_K else "both miss (in top 10)"

summary = pd.DataFrame([{
    "qid": r["qid"], "category": r["category"], "query": r["query"],
    "critical chunk: dense rank -> rerank rank": ", ".join(f"{c}: {r['critical_dense_rank'][c]} -> {r['critical_rerank_rank'][c] or '—'}"
                                                         for c in r["critical"]),
    "CE score": ", ".join(f"{c}: {s}" for c, s in r["critical_ce"].items()) or "—",
    "baseline top3": " ".join(c["chunk_id"] for c in r["base_top"]),
    "rerank top3": " ".join(c["chunk_id"] for c in r["rr_top"]),
    "critical in baseline top3": r["base_hit"], "critical in rerank top3": r["rr_hit"], "outcome": outcome(r)} for r in runs])
pd.DataFrame(rank_rows).to_csv(RESULTS_DIR / "rankings_top10.csv", index=False)
summary.to_csv(RESULTS_DIR / "rank_shift_summary.csv", index=False)
summary

,qid,category,query,critical chunk: dense rank -> rerank rank,CE score,baseline top3,rerank top3,critical in baseline top3,critical in rerank top3,outcome
0,Q01,keyword trap,Are password hints allowed?,C046: 6 -> 3,C046: -3.91,C198 C243 C244,C249 C244 C046,False,True,RE-RANK RESCUE
1,Q02,keyword trap,How often should users be forced to change their passwords?,"C049: 19 -> —, C198: 6 -> 1",C198: -0.5,C246 C251 C247,C198 C197 C247,False,True,RE-RANK RESCUE
2,Q03,keyword trap,What is the minimum password length?,"C043: 11 -> —, C044: 15 -> —",—,C246 C248 C245,C246 C096 C248,False,False,both miss (not in top 10)
3,Q04,keyword trap,Can SMS text messages be used to deliver one-time codes?,"C061: 8 -> 2, C062: 11 -> —, C068: 31 -> —",C061: -9.78,C200 C138 C213,C167 C061 C193,False,True,RE-RANK RESCUE
4,Q05,exception,How long is an out-of-band authentication secret valid?,C066: 4 -> 1,C066: 6.49,C065 C060 C061,C066 C058 C064,False,True,RE-RANK RESCUE
5,Q06,exception,Is a 6-character memorized secret ever acceptable?,"C043: 1 -> 1, C046: 3 -> 2","C043: 5.06, C046: 3.84",C043 C198 C046,C043 C046 C252,True,True,both hit
6,Q07,exception,How many failed fingerprint attempts are allowed before biometric authentication is disabled?,C107: 2 -> 1,C107: 7.19,C097 C107 C092,C107 C082 C097,True,True,both hit
7,Q08,exception,Which authenticators are permitted at AAL3?,C028: 10 -> 2,C028: 7.58,C027 C030 C006,C018 C028 C027,False,True,RE-RANK RESCUE
8,Q09,negation,Are hashed low-complexity passwords considered verifier-compromise resistant?,C117: 1 -> 1,C117: 8.5,C117 C116 C115,C117 C116 C115,True,True,both hit
9,Q10,negation,Does unlocking my phone with a PIN count as an authentication factor?,C022: 1 -> 1,C022: 1.5,C022 C042 C092,C022 C023 C062,True,True,both hit


In [13]:
print("Hit rate = fraction of queries whose critical chunk is among the 3 chunks given to the LLM")
print(f"  baseline : {summary['critical in baseline top3'].sum()}/{len(summary)}")
print(f"  re-rank  : {summary['critical in rerank top3'].sum()}/{len(summary)}")
print(f"  ceiling  : {(pd.Series([r['best_dense_rank'] for r in runs]) <= TOP_K).sum()}/{len(summary)}  (critical chunk inside the dense top 10 at all)")
print(summary["outcome"].value_counts().to_string())

Hit rate = fraction of queries whose critical chunk is among the 3 chunks given to the LLM
  baseline : 9/16
  re-rank  : 14/16
  ceiling  : 15/16  (critical chunk inside the dense top 10 at all)
outcome
both hit                     8
RE-RANK RESCUE               6
both miss (not in top 10)    1
RE-RANK LOSS                 1


### 8.2 Full Top-10 tables for the failure cases

For each query where the outcome differs between the pipelines, the complete candidate list with both ranks. `★` marks a critical chunk.

In [14]:
def top10_table(r):
    rows = []
    for c in sorted(r["ranked"], key=lambda c: c["dense_rank"]):
        ch = chunk_by_id[c["chunk_id"]]
        rows.append({"": "★" if c["chunk_id"] in r["critical"] else "", "chunk": c["chunk_id"], "section": ch["section"][:45],
                     "page": ch["page_start"], "dense rank": c["dense_rank"], "cosine": round(c["dense_score"], 3),
                     "CE score": round(c["ce_score"], 2), "rerank rank": c["rerank_rank"],
                     "shift": f"#{c['dense_rank']} → #{c['rerank_rank']}", "text": ch["text"][:90] + "…"})
    return pd.DataFrame(rows)

from IPython.display import display, Markdown
interesting = [r for r in runs if outcome(r) in ("RE-RANK RESCUE", "RE-RANK LOSS")]
for r in interesting:
    display(Markdown(f"**{r['qid']} ({r['category']}) – {outcome(r)}:** _{r['query']}_"))
    display(top10_table(r))

**Q01 (keyword trap) – RE-RANK RESCUE:** _Are password hints allowed?_

,,chunk,section,page,dense rank,cosine,CE score,rerank rank,shift,text
0,,C198,10.2.1 Memorized Secrets,64,1,0.570,-5.85,8,#1 → #8,"Encourage users to make memorized secrets as lengthy as they want, using any characters th…"
1,,C243,Appendix A— Strength of Memorized Secrets,78,2,0.557,-5.31,6,#2 → #6,Appendix A— Strength of Memorized Secrets This appendix is informative. Throughout this ap…
2,,C244,A.1 Introduction,78,3,0.554,-3.60,2,#3 → #2,A.1 Introduction Despite widespread frustration with the use of passwords from both a usab…
3,,C042,5.1 Requirements by Authenticator Type,24,4,0.546,-7.47,9,#4 → #9,5.1 Requirements by Authenticator Type 5.1.1 Memorized Secrets A Memorized Secret authenti…
4,,C251,A.3 Complexity,79,5,0.535,-4.14,4,#5 → #4,"For this reason, it is recommended that passwords chosen by users be compared against a “b…"
5,★,C046,5.1.1.2 Memorized Secret Verifiers,24,6,0.526,-3.91,3,#6 → #3,Subscribers choosing memorized secrets containing Unicode characters SHOULD be advised tha…
6,,C199,10.2.2 Look-Up Secrets,65,7,0.516,-10.72,10,#7 → #10,10.2.2 Look-Up Secrets Typical Usage Users use the authenticator — printed or electronic —…
7,,C050,5.1.1.2 Memorized Secret Verifiers,25,8,0.510,-4.96,5,#8 → #5,This allows the claimant to verify their entry if they are in a location where their scree…
8,,C197,10.2.1 Memorized Secrets,64,9,0.510,-5.69,7,#9 → #7,10.2.1 Memorized Secrets Typical Usage Users manually input the memorized secret (commonly…
9,,C249,A.3 Complexity,79,10,0.502,-2.97,1,#10 → #1,"A.3 Complexity As noted above, composition rules are commonly used in an attempt to increa…"


**Q02 (keyword trap) – RE-RANK RESCUE:** _How often should users be forced to change their passwords?_

,,chunk,section,page,dense rank,cosine,CE score,rerank rank,shift,text
0,,C246,A.2 Length,78,1,0.497,-6.00,8,#1 → #8,A.2 Length Password length has been found to be a primary factor in characterizing passwor…
1,,C251,A.3 Complexity,79,2,0.452,-3.18,4,#2 → #4,"For this reason, it is recommended that passwords chosen by users be compared against a “b…"
2,,C247,A.2 Length,78,3,0.446,-2.66,3,#3 → #3,Offline attacks are sometimes possible when one or more hashed passwords is obtained by th…
3,,C223,11.1 General References,73,4,0.444,-9.68,10,#4 → #10,"[Composition] Komanduri, Saranga, Richard Shay, Patrick Gage Kelley, Michelle L Mazurek, L…"
4,,C244,A.1 Introduction,78,5,0.435,-3.71,6,#5 → #6,A.1 Introduction Despite widespread frustration with the use of passwords from both a usab…
5,★,C198,10.2.1 Memorized Secrets,64,6,0.429,-0.50,1,#6 → #1,"Encourage users to make memorized secrets as lengthy as they want, using any characters th…"
6,,C228,11.1 General References,74,7,0.410,-9.27,9,#7 → #9,Available at: http://research.microsoft.com/apps/pubs/default.aspx?id=154077. [Privacy Act…
7,,C248,A.2 Length,79,8,0.403,-3.49,5,#8 → #5,"Since the size of a hashed password is independent of its length, there is no reason not t…"
8,,C245,A.1 Introduction,78,9,0.397,-5.96,7,#9 → #7,Complexity of user-chosen passwords has often been characterized using the information the…
9,,C197,10.2.1 Memorized Secrets,64,10,0.394,-2.33,2,#10 → #2,10.2.1 Memorized Secrets Typical Usage Users manually input the memorized secret (commonly…


**Q04 (keyword trap) – RE-RANK RESCUE:** _Can SMS text messages be used to deliver one-time codes?_

...

,,chunk,section,page,dense rank,cosine,CE score,rerank rank,shift,text
0,,C027,4.3 Authenticator Assurance Level 3,19,1,0.800,7.02,3,#1 → #3,4.3 Authenticator Assurance Level 3 This section is normative. AAL3 provides very high con…
1,,C030,4.3.2 Authenticator and Verifier Requirements,20,2,0.759,6.71,5,#2 → #5,4.3.2 Authenticator and Verifier Requirements Communication between the claimant and verif…
2,,C006,2 Introduction,14,3,0.756,6.52,6,#3 → #6,AAL3 authentication requires a hardware-based authenticator and an authenticator that prov…
3,,C018,4.2.1 Permitted Authenticator Types,18,4,0.755,8.24,1,#4 → #1,"4.2.1 Permitted Authenticator Types At AAL2, authentication SHALL occur by the use of eith…"
4,,C041,5 Authenticator and Verifier Requirements,24,5,0.753,5.91,7,#5 → #7,5 Authenticator and Verifier Requirements This section is normative. This section provides…
5,,C005,2 Introduction,13,6,0.751,5.07,9,#6 → #9,Successful authentication requires that the claimant prove possession and control of the a…
6,,C021,4.2.2 Authenticator and Verifier Requirements,18,7,0.737,6.71,4,#7 → #4,4.2.2 Authenticator and Verifier Requirements Cryptographic authenticators used at AAL2 SH…
7,,C017,4.2 Authenticator Assurance Level 2,17,8,0.729,5.65,8,#8 → #8,4.2 Authenticator Assurance Level 2 This section is normative. AAL2 provides high confiden…
8,,C011,4.1 Authenticator Assurance Level 1,16,9,0.729,4.29,10,#9 → #10,4.1 Authenticator Assurance Level 1 This section is normative. AAL1 provides some assuranc…
9,★,C028,4.3.1 Permitted Authenticator Types,20,10,0.727,7.58,2,#10 → #2,4.3.1 Permitted Authenticator Types AAL3 authentication SHALL occur by the use of one of a…


**Q13 (role/level limit) – RE-RANK LOSS:** _Can a password alone satisfy AAL2?_

,,chunk,section,page,dense rank,cosine,CE score,rerank rank,shift,text
0,,C017,4.2 Authenticator Assurance Level 2,17,1,0.588,-0.90,3,#1 → #3,4.2 Authenticator Assurance Level 2 This section is normative. AAL2 provides high confiden…
1,★,C018,4.2.1 Permitted Authenticator Types,18,2,0.566,-2.18,8,#2 → #8,"4.2.1 Permitted Authenticator Types At AAL2, authentication SHALL occur by the use of eith…"
2,,C005,2 Introduction,13,3,0.553,-0.61,2,#3 → #2,Successful authentication requires that the claimant prove possession and control of the a…
3,,C022,4.2.2 Authenticator and Verifier Requirements,18,4,0.553,0.52,1,#4 → #1,Authentication at AAL2 SHOULD demonstrate authentication intent from at least one authenti…
4,,C021,4.2.2 Authenticator and Verifier Requirements,18,5,0.548,-1.37,4,#5 → #4,4.2.2 Authenticator and Verifier Requirements Cryptographic authenticators used at AAL2 SH…
5,,C011,4.1 Authenticator Assurance Level 1,16,6,0.530,-2.51,9,#6 → #9,4.1 Authenticator Assurance Level 1 This section is normative. AAL1 provides some assuranc…
6,,C012,4.1.1 Permitted Authenticator Types,16,7,0.528,-1.96,6,#7 → #6,4.1.1 Permitted Authenticator Types AAL1 authentication SHALL occur by the use of any of t…
7,,C004,2 Introduction,13,8,0.521,-2.15,7,#8 → #7,The strength of an authentication transaction is characterized by an ordinal measurement k…
8,,C027,4.3 Authenticator Assurance Level 3,19,9,0.513,-1.61,5,#9 → #5,4.3 Authenticator Assurance Level 3 This section is normative. AAL3 provides very high con…
9,,C003,2 Introduction,13,10,0.502,-5.02,10,#10 → #10,"A successful authentication results in the assertion of an identifier, either pseudonymous…"


**Q15 (buried detail) – RE-RANK RESCUE:** _Can push notifications on a smartphone be used for out-of-band authentication?_

,,chunk,section,page,dense rank,cosine,CE score,rerank rank,shift,text
0,,C200,10.2.3 Out-of-Band,65,1,0.571,2.62,3,#1 → #3,10.2.3 Out-of-Band Typical Usage Out-of-band authentication requires users have access to …
1,,C058,5.1.3 Out-of-Band Devices,27,2,0.536,0.76,8,#2 → #8,5.1.3 Out-of-Band Devices An out-of-band authenticator is a physical device that is unique…
2,,C201,10.2.3 Out-of-Band,65,3,0.527,2.91,2,#3 → #2,Providing users such features is particularly helpful when the primary and secondary chann…
3,,C061,5.1.3.1 Out-of-Band Authenticators,28,4,0.514,2.37,5,#4 → #5,"Methods that do not prove possession of a specific device, such as voice-over-IP (VOIP) or…"
4,★,C064,5.1.3.2 Out-of-Band Verifiers,29,5,0.501,6.28,1,#5 → #1,5.1.3.2 Out-of-Band Verifiers For additional verification requirements specific to the PST…
5,,C060,5.1.3.1 Out-of-Band Authenticators,28,6,0.495,2.50,4,#6 → #4,5.1.3.1 Out-of-Band Authenticators The out-of-band authenticator SHALL establish a separat…
6,,C062,5.1.3.1 Out-of-Band Authenticators,28,7,0.487,1.54,6,#7 → #6,This method SHALL only be used if a secret is being sent from the verifier to the out-of-b…
7,,C022,4.2.2 Authenticator and Verifier Requirements,18,8,0.481,1.43,7,#8 → #7,Authentication at AAL2 SHOULD demonstrate authentication intent from at least one authenti…
8,,C065,5.1.3.2 Out-of-Band Verifiers,29,9,0.445,-0.71,9,#9 → #9,"Depending on the type of out-of-band authenticator, one of the following SHALL take place:…"
9,,C119,5.2.9 Authentication Intent,41,10,0.437,-8.84,10,#10 → #10,5.2.9 Authentication Intent An authentication process demonstrates intent if it requires t…


### 8.3 Answer 1 (baseline) vs. Answer 2 (re-ranked)

Both answers are generated for every query. When both pipelines select the same three chunks (in any order) the prompts differ only in passage order.

In [15]:
answers = []
for r in runs:
    p1 = build_prompt(r["query"], r["base_top"])
    p2 = build_prompt(r["query"], r["rr_top"])
    a1 = generate(p1)
    a2 = generate(p2)
    chk = lambda a: None if a["text"] is None else bool(re.search(r["answer_check"], a["text"]))
    answers.append({"qid": r["qid"], "query": r["query"], "outcome": outcome(r),
                    "same_chunk_set": {c["chunk_id"] for c in r["base_top"]} == {c["chunk_id"] for c in r["rr_top"]},
                    "answer1": a1["text"], "answer2": a2["text"],
                    "answer1_has_key_fact": chk(a1), "answer2_has_key_fact": chk(a2),
                    "llm_latency1_s": a1["latency_s"], "llm_latency2_s": a2["latency_s"],
                    "model_version": a1.get("model_version") or a2.get("model_version"),
                    "from_cache": [a1.get("cached"), a2.get("cached")],
                    "errors": [e for e in (a1.get("error"), a2.get("error")) if e]})

with open(RESULTS_DIR / "answers.json", "w") as f:
    json.dump(answers, f, indent=2, ensure_ascii=False)
ans_df = pd.DataFrame(answers)
print("LLM model version reported by API:", ans_df["model_version"].dropna().unique(), "| failed calls:", int(ans_df["errors"].map(len).sum()))
ans_df[["qid", "outcome", "same_chunk_set", "answer1_has_key_fact", "answer2_has_key_fact"]]

LLM model version reported by API: <ArrowStringArray>
['gemini-3.1-flash-lite']
Length: 1, dtype: str | failed calls: 0


,qid,outcome,same_chunk_set,answer1_has_key_fact,answer2_has_key_fact
0,Q01,RE-RANK RESCUE,False,False,True
1,Q02,RE-RANK RESCUE,False,False,True
2,Q03,both miss (not in top 10),False,False,False
3,Q04,RE-RANK RESCUE,False,False,False
4,Q05,RE-RANK RESCUE,False,False,True
5,Q06,both hit,False,True,True
6,Q07,both hit,False,True,True
7,Q08,RE-RANK RESCUE,False,False,True
8,Q09,both hit,True,True,True
9,Q10,both hit,False,True,True


In [16]:
# Print answers side by side for the queries where the selected chunks differ.
for a in answers:
    if a["same_chunk_set"]:
        continue
    display(Markdown(f"---\n**{a['qid']} – {a['outcome']}** — _{a['query']}_\n\n"
                     f"**Answer 1 (baseline):** {a['answer1']}\n\n**Answer 2 (re-ranked):** {a['answer2']}"))

---
**Q01 – RE-RANK RESCUE** — _Are password hints allowed?_

**Answer 1 (baseline):** The provided context does not answer this question.

**Answer 2 (re-ranked):** Memorized secret verifiers SHALL NOT permit the subscriber to store a “hint” that is accessible to an unauthenticated claimant [C046].

---
**Q02 – RE-RANK RESCUE** — _How often should users be forced to change their passwords?_

**Answer 1 (baseline):** The provided context does not answer this question.

**Answer 2 (re-ranked):** Users SHOULD NOT be required to change their memorized secrets arbitrarily (e.g., periodically) unless there is a user request or evidence of authenticator compromise [C198].

---
**Q03 – both miss (not in top 10)** — _What is the minimum password length?_

**Answer 1 (baseline):** The provided context does not answer this question.

**Answer 2 (re-ranked):** The provided context does not answer this question.

---
**Q04 – RE-RANK RESCUE** — _Can SMS text messages be used to deliver one-time codes?_

**Answer 1 (baseline):** The provided context does not answer this question.

**Answer 2 (re-ranked):** Yes, SMS text messages can be used to deliver one-time codes as part of an out-of-band authentication method, provided the method authenticates to a public mobile telephone network using a SIM card or equivalent that uniquely identifies the device [C061]. This method SHALL only be used if a secret is being sent from the verifier to the out-of-band device via the PSTN (SMS or voice) [C061].

---
**Q05 – RE-RANK RESCUE** — _How long is an out-of-band authentication secret valid?_

**Answer 1 (baseline):** The provided context does not answer this question.

**Answer 2 (re-ranked):** The authentication SHALL be considered invalid if not completed within 10 minutes [C066]. Additionally, verifiers SHALL accept a given authentication secret only once during the validity period to provide replay resistance [C066].

...

---
**Q11 – both hit** — _Is email an acceptable out-of-band authentication method?_

**Answer 1 (baseline):** No, email is not an acceptable out-of-band authentication method. Methods that do not prove possession of a specific device, such as email, SHALL NOT be used for out-of-band authentication [C060], [C061].

**Answer 2 (re-ranked):** No, email is not an acceptable out-of-band authentication method. Methods that do not prove possession of a specific device, such as email or voice-over-IP (VOIP), SHALL NOT be used for out-of-band authentication [C060], [C061].

---
**Q12 – both hit** — _Can session cookies survive a browser restart?_

**Answer 1 (baseline):** Session secrets, which include session cookies, SHALL be non-persistent [C156]. Therefore, they SHALL NOT be retained across a restart of the associated application or a reboot of the host device [C156].

**Answer 2 (re-ranked):** No, session secrets, which include browser cookies used for session tracking, SHALL be non-persistent [C156]. They SHALL NOT be retained across a restart of the associated application or a reboot of the host device [C156].

---
**Q13 – RE-RANK LOSS** — _Can a password alone satisfy AAL2?_

**Answer 1 (baseline):** No, a password alone cannot satisfy AAL2. At AAL2, authentication SHALL occur by the use of either a multi-factor authenticator or a combination of two single-factor authenticators [C018]. Furthermore, proof of possession and control of two distinct authentication factors is required at AAL2 [C017], [C005].

**Answer 2 (re-ranked):** No, a password alone cannot satisfy AAL2. AAL2 requires proof of possession and control of two distinct authentication factors through secure authentication protocol(s) [C005, C017].

---
**Q15 – RE-RANK RESCUE** — _Can push notifications on a smartphone be used for out-of-band authentication?_

**Answer 1 (baseline):** The provided context does not answer this question.

**Answer 2 (re-ranked):** Yes, if out-of-band verification is to be made using a secure application, such as on a smartphone, the verifier MAY send a push notification to that device [C064]. The verifier then waits for the establishment of an authenticated protected channel and verifies the authenticator’s identifying key [C064].

---
**Q16 – both hit** — _What false match rate must a biometric system achieve?_

**Answer 1 (baseline):** The biometric system SHALL operate with an FMR of 1 in 1000 or better [C106]. This FMR SHALL be achieved under conditions of a conformant attack (i.e., zero-effort impostor attempt) as defined in ISO/IEC 30107-1 [C106].

**Answer 2 (re-ranked):** The biometric system SHALL operate with an FMR of 1 in 1000 or better [C106]. This FMR SHALL be achieved under conditions of a conformant attack (i.e., zero-effort impostor attempt) as defined in ISO/IEC 30107-1 [C106].

## 9. Latency profiling

**Measurement boundaries** (wall-clock, `time.perf_counter`, CPU only, models already loaded, after warm-up):

* **Baseline retrieval** = encode query with bi-encoder + FAISS search for Top 10 + take first 3.
* **Re-rank pipeline retrieval** = the same dense step **plus** cross-encoder scoring of the 10 pairs + sort + take first 3.
* **LLM generation** is timed separately (one Gemini call per answer, network + server time, measured when the call was made and stored in the cache) because it is dominated by the API and is the same kind of work in both pipelines.

Each query is timed `REPEATS` times; the table reports the median and 95th percentile over all query × repeat measurements.

In [17]:
REPEATS = 20

def time_once(q):
    t0 = time.perf_counter()
    cands = dense_retrieve(q); _ = baseline_select(cands)
    t1 = time.perf_counter()
    _ = rerank(q, cands)
    t2 = time.perf_counter()
    return t1 - t0, t2 - t1

for qd in QUERIES[:3]:                              # warm-up (thread pools, caches)
    time_once(qd["query"])

dense_t, ce_t = [], []
for _ in range(REPEATS):
    for qd in QUERIES:
        d, c = time_once(qd["query"])
        dense_t.append(d); ce_t.append(c)
dense_ms, ce_ms = np.array(dense_t) * 1000, np.array(ce_t) * 1000
total_ms = dense_ms + ce_ms

llm1 = [a["llm_latency1_s"] for a in answers if a["llm_latency1_s"] is not None]
llm2 = [a["llm_latency2_s"] for a in answers if a["llm_latency2_s"] is not None]
stat = lambda x: (round(float(np.median(x)), 1), round(float(np.percentile(x, 95)), 1), round(float(np.mean(x)), 1))
rows = [
    ("Pipeline 1: dense retrieval (encode + FAISS top-10 + top-3)", *stat(dense_ms), len(dense_ms)),
    ("Re-ranking step only (cross-encoder on 10 pairs + sort)",      *stat(ce_ms), len(ce_ms)),
    ("Pipeline 2: dense retrieval + re-ranking",                     *stat(total_ms), len(total_ms)),
]
if llm1 and llm2:
    rows += [("LLM generation, Answer 1 (Gemini API)", *stat(np.array(llm1) * 1000), len(llm1)),
             ("LLM generation, Answer 2 (Gemini API)", *stat(np.array(llm2) * 1000), len(llm2))]
latency = pd.DataFrame(rows, columns=["stage", "median ms", "p95 ms", "mean ms", "n"])
latency.to_csv(RESULTS_DIR / "latency.csv", index=False)
print(f"Re-ranking adds {np.median(ce_ms):.1f} ms median ({np.median(total_ms) / np.median(dense_ms):.1f}x the baseline retrieval time).")
latency

Re-ranking adds 183.3 ms median (23.4x the baseline retrieval time).


,stage,median ms,p95 ms,mean ms,n
0,Pipeline 1: dense retrieval (encode + FAISS top-10 + top-3),8.2,11.2,8.5,320
1,Re-ranking step only (cross-encoder on 10 pairs + sort),183.3,250.5,189.8,320
2,Pipeline 2: dense retrieval + re-ranking,191.6,261.6,198.3,320
3,"LLM generation, Answer 1 (Gemini API)",1648.7,2670.0,1713.8,16
4,"LLM generation, Answer 2 (Gemini API)",1433.8,1948.8,1483.9,16


### Scaling: why not run the cross-encoder over the whole collection?

To ground the scaling question in measurements, the cross-encoder is timed on all chunks of this document for one query and compared with scoring 10 candidates. The per-pair cost is then extrapolated linearly to larger collections (an **extrapolation**, labelled as such). The bi-encoder side only needs one query embedding at query time because chunk embeddings were computed offline (§4).

In [18]:
q = QUERIES[0]["query"]
all_pairs = [(q, c["text"]) for c in chunks]
cross_encoder.predict(all_pairs[:32], batch_size=32, show_progress_bar=False)       # warm-up
full_times = []
for _ in range(3):
    t0 = time.perf_counter(); cross_encoder.predict(all_pairs, batch_size=32, show_progress_bar=False)
    full_times.append(time.perf_counter() - t0)
full_s = float(np.median(full_times))
per_pair_ms = full_s * 1000 / len(all_pairs)
emb_per_chunk_ms = index_build_s * 1000 / len(chunks)

scale = pd.DataFrame([{
    "collection size (chunks)": n,
    "CE on all chunks per query (s)": round(n * per_pair_ms / 1000, 2),
    "dense search + CE on top-10 per query (ms)": round(float(np.median(total_ms)), 1) if n == len(chunks) else "≈ same (+ ANN search)",
    "one-off offline embedding cost (s)": round(n * emb_per_chunk_ms / 1000, 1),
} for n in [len(chunks), 10_000, 1_000_000]])
print(f"measured: CE over all {len(chunks)} chunks = {full_s*1000:.0f} ms per query ({per_pair_ms:.2f} ms/pair); "
      f"CE over 10 candidates = {np.median(ce_ms):.1f} ms")
scale.to_csv(RESULTS_DIR / "scaling.csv", index=False)
scale

measured: CE over all 254 chunks = 4781 ms per query (18.82 ms/pair); CE over 10 candidates = 183.3 ms


,collection size (chunks),CE on all chunks per query (s),dense search + CE on top-10 per query (ms),one-off offline embedding cost (s)
0,254,4.78,191.6,6.4
1,10000,188.23,≈ same (+ ANN search),253.3
2,1000000,18823.09,≈ same (+ ANN search),25328.5


## 10. Conclusions and limitations

The cell below prints the measured numbers this section is based on; the interpretation is in the report (`report/report.md`), which references these saved results (`results/`).

**Limitations (apply to every result above):**
* Small evaluation set (16 hand-written queries on one document) – the counts are descriptive, not statistically significant.
* Relevance labels are key-phrase matches chosen by me; a chunk can be relevant without containing the phrase (and vice versa). The 'answer_check' regex is a coarse screen and the answers were also read manually.
* Queries were written after reading the document and refined while exploring retrieval, so they are targeted stress tests, not a random sample of real user questions.
* The re-ranker can only reorder what the bi-encoder retrieved: when the critical chunk is outside the dense Top 10, neither pipeline can recover it.
* Both encoders are small MiniLM models (fast on CPU, not state of the art). Gemini output can vary slightly between runs even at temperature 0; answers are cached in `results/llm_cache.json`, so re-running the notebook reproduces the committed answers (delete the cache to regenerate).
* Latency was measured on one CPU laptop; LLM latency depends on network and API load.

In [19]:
print(summary[["qid", "category", "outcome"]].to_string(index=False))
print()
print(latency.to_string(index=False))

qid         category                   outcome
Q01     keyword trap            RE-RANK RESCUE
Q02     keyword trap            RE-RANK RESCUE
Q03     keyword trap both miss (not in top 10)
Q04     keyword trap            RE-RANK RESCUE
Q05        exception            RE-RANK RESCUE
Q06        exception                  both hit
Q07        exception                  both hit
Q08        exception            RE-RANK RESCUE
Q09         negation                  both hit
Q10         negation                  both hit
Q11         negation                  both hit
Q12         negation                  both hit
Q13 role/level limit              RE-RANK LOSS
Q14 role/level limit                  both hit
Q15    buried detail            RE-RANK RESCUE
Q16    buried detail                  both hit

                                                      stage  median ms  p95 ms  mean ms   n
Pipeline 1: dense retrieval (encode + FAISS top-10 + top-3)        8.2    11.2      8.5 320
    Re-ranking s